In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

project_dir = Path.cwd()
data_file = project_dir / 'data.csv'
output_file = project_dir / 'data_clean.csv'

print(f"Projektverzeichnis: {project_dir}")
print(f"Eingabedatei: {data_file}")

df_raw = pd.read_csv(data_file)

print("\nRohdaten-Übersicht:")
print(f"Shape: {df_raw.shape}")
print(f"Spalten: {df_raw.columns.tolist()}")
print(df_raw.head())

# Filtern der relevanten Messwerte nach Farbe und Messgröße
df_filtered = df_raw[df_raw['event_type'].isin(['dispenser_red', 'dispenser_blue', 'dispenser_green', 'temperature'])].copy()

# Farbe aus event_type extrahieren
def get_color_from_event(row):
    if pd.notna(row['dispenser']) and row['dispenser'] in ['red', 'blue', 'green']:
        return row['dispenser']
    event_type = row['event_type']
    if 'red' in event_type:
        return 'red'
    elif 'blue' in event_type:
        return 'blue'
    elif 'green' in event_type:
        return 'green'
    return None

df_filtered['color'] = df_filtered.apply(get_color_from_event, axis=1)

# Aggregation der Messwerte pro Flasche und Farbe
df_wide = df_filtered.groupby(['bottle', 'color']).agg({
    'fill_level_grams': 'mean',
    'vibration_index': 'mean',
    'temperature_C': 'mean'
}).reset_index()

# Nur Flaschen mit Endgewicht berücksichtigen
valid_bottles = df_raw[df_raw['final_weight'].notna()][['bottle', 'final_weight']].drop_duplicates('bottle')

# Pivotieren ins Wide-Format
df_pivot = pd.pivot_table(
    df_wide,
    index='bottle',
    columns='color',
    values=['fill_level_grams', 'vibration_index', 'temperature_C'],
    aggfunc='mean'
)

df_pivot.columns = [f'{metric}_{color}' for metric, color in df_pivot.columns]

# Endgewicht hinzufügen
df_pivot = df_pivot.reset_index()
df_pivot = df_pivot.merge(valid_bottles, on='bottle', how='inner')

print("\nDaten in Wide-Format transformiert!")
print(f"Shape: {df_pivot.shape}")
print(f"Spalten: {df_pivot.columns.tolist()}")
print(df_pivot.head())

df_pivot.to_csv(output_file, index=False)
print(f"\nBereinigte Daten als '{output_file}' gespeichert")

Projektverzeichnis: c:\Users\user\Documents\a_MCI\4_Semester\Automatisierungstechnik\IIOT_Automatisierungstechnik\Aufgabe_12.3
Eingabedatei: c:\Users\user\Documents\a_MCI\4_Semester\Automatisierungstechnik\IIOT_Automatisierungstechnik\Aufgabe_12.3\data.csv

Rohdaten-Übersicht:
Shape: (6416, 15)
Spalten: ['timestamp', 'event_type', 'bottle', 'dispenser', 'time', 'temperature_C', 'fill_level_grams', 'recipe', 'vibration_index', 'final_weight', 'is_cracked', 'drop_oscillation', 'id', 'creation_date', 'color_levels_grams']
                    timestamp      event_type      bottle dispenser  \
0  2026-06-05T09:31:10.171314          recipe         NaN       NaN   
1  2026-06-05T09:31:12.009450   dispenser_red  80582707.0       red   
2  2026-06-05T09:31:12.067740     temperature         NaN       red   
3  2026-06-05T09:31:12.072093  dispenser_blue  80582705.0      blue   
4  2026-06-05T09:31:12.077170     temperature         NaN      blue   

           time  temperature_C  fill_level_grams

# Automatisierungstechnik - Flaschengewicht Vorhersage

## 1. Datenbereinigung und Transformation
Konvertierung der Rohdaten vom Long-Format in das Wide-Format für das Regressionsmodell.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from pathlib import Path

project_dir = Path(r'c:\Users\user\Documents\a_MCI\4_Semester\Automatisierungstechnik\Prediction')
data_clean_file = project_dir / 'data_clean.csv'
x_file = project_dir / 'X.csv'

print(f"Projektverzeichnis: {project_dir}")

df = pd.read_csv(data_clean_file)

print("\nTrainingsdaten geladen:")
print(f"Shape: {df.shape}")
print(f"Verfügbare Spalten: {df.columns.tolist()}\n")

# Feature-Sets zum Vergleichen
features_set_1 = ['fill_level_grams_red', 'fill_level_grams_blue', 'fill_level_grams_green']
features_set_2 = features_set_1 + ['vibration_index_red', 'vibration_index_blue', 'vibration_index_green']

feature_sets = {
    "Nur Füllstand": features_set_1,
    "Füllstand + Vibration": features_set_2
}

target = 'final_weight'

results = []
best_model = None
best_features = None
lowest_mse_test = float('inf')

print("=" * 60)
print("--- Modell-Evaluation ---")
print("=" * 60)

for name, features in feature_sets.items():
    print(f"\nTrainiere Modell: {name}")
    print(f"   Spalten: {features}")
    
    df_clean = df.dropna(subset=features + [target])
    print(f"   Trainingsproben (nach Löschen von NaN): {len(df_clean)}")
    
    X = df_clean[features]
    y = df_clean[target]
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    
    mse_train = mean_squared_error(y_train, y_pred_train)
    mse_test = mean_squared_error(y_test, y_pred_test)
    
    print(f"   MSE (Training): {mse_train:.4f}")
    print(f"   MSE (Test):     {mse_test:.4f}")
    
    results.append({
        "Genutzte Spalten (X)": str(features),
        "Modell-Typ": "Linear",
        "MSE (Training)": round(mse_train, 4),
        "MSE (Test)": round(mse_test, 4)
    })
    
    if mse_test < lowest_mse_test:
        lowest_mse_test = mse_test
        best_model = model
        best_features = features

print("\n" + "=" * 60)
print("ERGEBNISSE")
print("=" * 60)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n" + "=" * 60)
print("BESTES MODELL")
print("=" * 60)
print(f"Genutzte Spalten (X): {results_df.loc[results_df['MSE (Test)'].idxmin(), 'Genutzte Spalten (X)']}")
print(f"MSE (Test): {results_df['MSE (Test)'].min():.4f}\n")

print("Formel (y = mx + b):")
intercept = best_model.intercept_
coefficients = best_model.coef_

formel = f"final_weight = {intercept:.4f}"
for coef, feat in zip(coefficients, best_features):
    sign = "+" if coef >= 0 else ""
    formel += f" {sign} {coef:.4f} * {feat}"
print(formel)

print("\n" + "=" * 60)
print("--- Prognose für X.csv ---")
print("=" * 60)
try:
    df_X = pd.read_csv(x_file)
    print(f"X.csv geladen: {df_X.shape[0]} Flaschen")
    
    X_unseen = df_X[best_features].fillna(0)
    y_hat = best_model.predict(X_unseen)
    
    id_column = 'Flaschen ID' if 'Flaschen ID' in df_X.columns else 'bottle'
    flaschen_id = df_X[id_column] if id_column in df_X.columns else df_X.index + 1
    
    ausgabe_df = pd.DataFrame({
        'Flaschen ID': flaschen_id,
        'y_hat': [round(val, 2) for val in y_hat]
    })
    
    gruppen_name = "eli_maxi_artur" 
    dateiname = f'reg_{gruppen_name}.csv'
    ausgabe_path = project_dir / dateiname
    ausgabe_df.to_csv(ausgabe_path, index=False)
    
    print(f"\nErfolg! Die Prognose wurde unter '{ausgabe_path}' gespeichert.")
    print(f"\nVorschau:")
    print(ausgabe_df.head(10).to_string(index=False))
    
except FileNotFoundError as e:
    print(f"FEHLER: {e}")
    print(f"Überprüfe, dass die Datei 'X.csv' in {project_dir} existiert.")
except KeyError as e:
    print(f"FEHLER: Eine benötigte Spalte fehlt in der CSV-Datei: {e}")

Projektverzeichnis: c:\Users\user\Documents\a_MCI\4_Semester\Automatisierungstechnik\Prediction

Trainingsdaten geladen:
Shape: (711, 8)
Verfügbare Spalten: ['bottle', 'fill_level_grams_blue', 'fill_level_grams_green', 'fill_level_grams_red', 'vibration_index_blue', 'vibration_index_green', 'vibration_index_red', 'final_weight']

--- Modell-Evaluation ---

Trainiere Modell: Nur Füllstand
   Spalten: ['fill_level_grams_red', 'fill_level_grams_blue', 'fill_level_grams_green']
   Trainingsproben (nach Löschen von NaN): 692
   MSE (Training): 54.8855
   MSE (Test):     42.5649

Trainiere Modell: Füllstand + Vibration
   Spalten: ['fill_level_grams_red', 'fill_level_grams_blue', 'fill_level_grams_green', 'vibration_index_red', 'vibration_index_blue', 'vibration_index_green']
   Trainingsproben (nach Löschen von NaN): 692
   MSE (Training): 0.1524
   MSE (Test):     0.1596

ERGEBNISSE
                                                                                                            